# Flight delay analysis - Data cleaning & transformation
_____

This script cleans the raw dataset produced by the scraping stage and
engineers the features needed for the EDA report: delay in minutes,
delay status, delay level (bucketed), scheduled datetime and time slot.

Expected raw columns (as scraped, in Spanish):
    - "fecha"               -> flight date
    - "Hora Programada"     -> scheduled time (string)
    - "Demora en despegar"  -> raw delay text, e.g. "Demora 15 min", "Cancelado"

These are renamed to English right after loading (see COLUMN_RENAME_MAP),
so everything downstream works with English column names.

## Constants

In [1]:
import io
import os
import re
from typing import Optional

import pandas as pd

In [2]:
COLUMN_RENAME_MAP = {
    "fecha": "date",
    "Hora Programada": "scheduled_time",
    "Demora en despegar": "delay_raw",
}

DELAY_LEVEL_LABELS = [
    "On Time/Early",
    "Minor Delay (0-15m)",
    "Moderate Delay (15-45m)",
    "Severe Delay (45m-3h)",
    "Critical (>3h)",
]

# Final categorical order, including the "Cancelled" bucket which doesn't
# come from the numeric bins (cancelled flights have no delay value).
DELAY_LEVEL_ORDER = DELAY_LEVEL_LABELS + ["Cancelled"]

DELAY_BINS = [-float("inf"), 0, 15, 45, 180, float("inf")]

# Spanish keywords used to classify the raw delay text scraped from the
# source site. These stay in Spanish on purpose: they match the language of
# the raw data, not the language of the codebase.
DELAY_KEYWORDS = {
    "cancelled": ["cancelado"],
    "on_time": ["a tiempo"],
    "early": ["adelantado"],
    "late": ["tarde"],
}

REQUIRED_RAW_COLUMNS = list(COLUMN_RENAME_MAP.keys())


## Helper functions

In [3]:
def load_data(file_path: str, **kwargs) -> pd.DataFrame:
    """
    Load a CSV or Parquet file into a pandas DataFrame.

    :param file_path: Path to the file.
    :param kwargs: Extra keyword arguments forwarded to pd.read_csv /
        pd.read_parquet.
    :return: Loaded DataFrame.
    :raises ValueError: If the file extension is not .csv or .parquet.
    :raises RuntimeError: If the file can't be read for any other reason.
    """
    _, file_extension = os.path.splitext(file_path)
    file_extension = file_extension.lower()

    if file_extension == ".csv":
        reader = pd.read_csv
    elif file_extension == ".parquet":
        reader = pd.read_parquet
    else:
        raise ValueError(f"Unsupported format: {file_extension}. Use CSV or Parquet.")

    try:
        return reader(file_path, **kwargs)
    except Exception as e:
        raise RuntimeError(f"Error loading file '{file_path}': {e}") from e


def validate_required_columns(df: pd.DataFrame, required_columns: list[str]) -> None:
    """
    Raise a clear error if the DataFrame is missing any expected column,
    instead of failing later with a confusing KeyError deep in the pipeline.
    """
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        raise KeyError(f"Missing expected column(s): {missing}. Found columns: {list(df.columns)}")


def print_report(title: str, data) -> None:
    """Print a labelled block of text/data with a simple ASCII header."""
    width = max(len(title), 25)
    print("=" * 40)
    print(title.upper())
    print("-" * width)
    print(data)
    print("=" * 40)


def basic_report_df(df: pd.DataFrame, title: str) -> None:
    """Print a quick overview of a DataFrame: shape, info, describe, nulls, uniques."""
    print("=" * 40)
    print("Report:\t", title)

    print_report("Shape:", f"Columns: {df.shape[1]}\t|\tRows: {df.shape[0]}")

    # df.info() prints directly and returns None, so we capture it into a
    # buffer instead of passing its (empty) return value to print_report.
    info_buffer = io.StringIO()
    df.info(buf=info_buffer)
    print_report("Info", info_buffer.getvalue())

    print_df(df.describe().T)

    print_report("Null values count:", df.isnull().sum())

    print_report("Unique values:", df.nunique())


def print_df(df_desc: pd.DataFrame) -> None:
    """Pretty-print a DataFrame using tabulate (requires `pip install tabulate`)."""
    from tabulate import tabulate

    # 'tablefmt' can be "grid", "fancy_grid", "pipe" or "psql"
    print(tabulate(df_desc, headers="keys", tablefmt="psql"))


## Transformation

In [4]:
def build_scheduled_datetime(
    df: pd.DataFrame,
    date_col: str,
    time_col: str,
    target_col: str,
) -> pd.DataFrame:
    """
    Combine a date column and a scheduled-time column into a single
    datetime column.

    :param date_col: Column holding the flight date.
    :param time_col: Column holding the scheduled time (string).
    :param target_col: Name of the new datetime column to create.
    """
    df[target_col] = pd.to_datetime(
        df[date_col].astype(str) + " " + df[time_col].astype(str),
        errors="coerce",
    )
    return df


def parse_delay(value: str, keywords: dict) -> tuple[Optional[int], str]:
    """
    Parse a raw delay string (in Spanish, e.g. "Demora 1hs 20min", "Cancelado",
    "A tiempo") into a (delay_in_minutes, status) tuple.

    :return: (minutes, status). minutes is None for cancelled flights,
        negative for early flights, 0 for on-time flights, and positive
        for delayed flights.
    """
    text = str(value).lower().strip()

    if any(k in text for k in keywords["cancelled"]):
        return None, "Cancelled"
    if any(k in text for k in keywords["on_time"]):
        return 0, "On time"

    hours_match = re.search(r"(\d+)\s*hs?", text)
    minutes_match = re.search(r"(\d+)\s*min", text)
    total_minutes = (int(hours_match.group(1)) * 60 if hours_match else 0) + (
        int(minutes_match.group(1)) if minutes_match else 0
    )

    if any(k in text for k in keywords["early"]):
        return -total_minutes, "Early"
    if any(k in text for k in keywords["late"]) or total_minutes > 0:
        return total_minutes, "Delayed"

    return 0, "On time"


def compute_delays(
    df: pd.DataFrame,
    source_col: str,
    delay_col: str = "net_delay_minutes",
    status_col: str = "status",
) -> pd.DataFrame:
    """Parse the raw delay text column into numeric minutes + status columns."""
    df[[delay_col, status_col]] = df[source_col].apply(
        lambda x: pd.Series(parse_delay(x, DELAY_KEYWORDS))
    )
    return df


def process_flights(
    df: pd.DataFrame,
    delay_raw_col: str = "delay_raw",
    date_col: str = "date",
    time_col: str = "scheduled_time",
) -> pd.DataFrame:
    """
    Run the full cleaning + feature engineering pipeline on a raw flights
    DataFrame: parses delays, flags cancellations, builds the scheduled
    datetime, extracts the hour slot, and buckets delays into levels.
    """
    validate_required_columns(df, [delay_raw_col, date_col, time_col])

    df[date_col] = pd.to_datetime(df[date_col])

    # 1. Parse delay text into minutes + status
    df = compute_delays(df, delay_raw_col, delay_col="net_delay_minutes", status_col="status")

    # 2. Cancellation flag
    df["is_cancelled"] = df[delay_raw_col].str.contains("Cancelado", na=False, case=False)

    # 3. Scheduled datetime + hour slot
    df = build_scheduled_datetime(df, date_col, time_col, target_col="scheduled_datetime")
    df["time_slot"] = df["scheduled_datetime"].dt.hour

    # 4. Bucket delays into levels. Cancelled flights have no numeric delay
    # (net_delay_minutes is NaN), so pd.cut leaves them as NaN; we then mark
    # those explicitly as "Cancelled" using a boolean mask instead of relying
    # on the implicit "nan"-string trick.
    is_cancelled_delay = df["net_delay_minutes"].isna()
    df["delay_level"] = pd.cut(
        df["net_delay_minutes"], bins=DELAY_BINS, labels=DELAY_LEVEL_LABELS
    ).astype(object)
    df.loc[is_cancelled_delay, "delay_level"] = "Cancelled"
    df["delay_level"] = pd.Categorical(
        df["delay_level"], categories=DELAY_LEVEL_ORDER, ordered=True
    )

    return df


def unify_datasets(datasets: list[pd.DataFrame], output_filename: str) -> pd.DataFrame:
    """Concatenate a list of DataFrames and save the result as a Parquet file."""
    df_final = pd.concat(datasets, ignore_index=True)
    output_path = f"{output_filename}.parquet"
    df_final.to_parquet(output_path)
    print(f"All datasets merged and saved to '{output_path}'!")
    return df_final



## Script entry point

In [6]:
def load_and_prepare_airline(file_path: str, report_title: str) -> pd.DataFrame:
    """Load a raw airline file, rename its columns to English, run a quick report."""
    df = load_data(file_path)
    validate_required_columns(df, REQUIRED_RAW_COLUMNS)
    df = df.rename(columns=COLUMN_RENAME_MAP)
    basic_report_df(df, report_title)
    return df


def main():
    # --- Import raw files ---
    df_jetsmart = load_and_prepare_airline(
        "vuelos_anual_WJ_consolidado_2025.parquet", "Jetsmart"
    )
    df_flybondi = load_and_prepare_airline(
        "vuelos_anual_FO_consolidado_2025.parquet", "Flybondi"
    )

    # --- Null values note ---
    # The "Hora Real" (actual time) column has 1000+ null values because
    # cancelled flights never have an actual departure time.

    # --- Cleaning & feature engineering ---
    df_jetsmart_final = process_flights(df_jetsmart)
    df_flybondi_final = process_flights(df_flybondi)

    print(df_jetsmart_final.dtypes)
    print(df_flybondi_final.dtypes)

    # --- Merge & save ---
    print("="*40)
    clean_datasets = [df_jetsmart_final, df_flybondi_final]
    unify_datasets(clean_datasets, "flights_historical_consolidated")

    # This script ends with a single consolidated dataset for both scraped
    # airlines, ready for the EDA report.


if __name__ == "__main__":
    main()

Report:	 Jetsmart
SHAPE:
-------------------------
Columns: 8	|	Rows: 23675
INFO
-------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23675 entries, 0 to 23674
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Vuelo           23675 non-null  object
 1   Ruta            23675 non-null  object
 2   scheduled_time  23675 non-null  object
 3   Hora Real       23334 non-null  object
 4   delay_raw       23675 non-null  object
 5   date            23675 non-null  object
 6   mes             23675 non-null  int64 
 7   empresa         23675 non-null  object
dtypes: int64(1), object(7)
memory usage: 1.4+ MB

+-----+---------+--------+---------+-------+-------+-------+-------+-------+
|     |   count |   mean |     std |   min |   25% |   50% |   75% |   max |
|-----+---------+--------+---------+-------+-------+-------+-------+-------|
| mes |   23675 | 6.8694 | 3.42739 |     1 |     4 |     7 |   